# Project 03 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs**, the headline one being an **omitted exposure offset**. Run it, read the diagnostics, find each bug, and fix it. The clean reference is `notebook.ipynb`; the answer key is `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240603

In [ ]:
from data.generate_data import generate
data = generate()
y, e = data['y'], data['exposure']

### Model — the exposure is ignored, and the rate prior is on the wrong scale.

In [ ]:
# BUG 1 (headline): the exposure offset is OMITTED. mu = exp(log_rate) for
#   every sample, ignoring that exposures vary over [5, 40]. This mistakes
#   'more exposure -> more counts' for 'a higher rate' and biases the estimate.
# BUG 2: a flat prior placed DIRECTLY on the positive rate (Uniform), instead
#   of a Normal prior on the LOG rate. This breaks the log-link parameterization
#   and puts implausible mass on huge rates.
with pm.Model() as model:
    rate = pm.Uniform('rate', lower=0.0, upper=1e4)   # BUG 2
    mu = rate                                          # BUG 1: no * exposure
    pm.Poisson('y', mu=mu, observed=y)
    idata = pm.sample(draws=1000, tune=1000, chains=2, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['rate']))
print('true rate =', np.exp(data['truth']['log_rate']),
      '-> the estimate will be ~mean count ~7, not ~0.30')

### Posterior predictive — BUG 3: a residual check summed over the wrong axis.

In [ ]:
with model:
    idata.extend(pm.sample_posterior_predictive(idata, random_seed=RNG,
                                                progressbar=False))
pp = idata.posterior_predictive['y']
# BUG 3: summing over 'draw' instead of the observation axis for the total count
pp_tot = pp.sum(dim='draw').values.ravel()
print('observed total =', y.sum(), 'predicted total mean =', pp_tot.mean())